## Does Calgary have transit deserts?
### Measuring access to Calgary's social hubs for kids

**by Faiza Rahman**

---

Public transit isn't equally accessible to everyone and as talks of regulating social media for youth increases, there is not much in regards to alternatives. Young people without cars depend on transit most, yet the stops aren't always where they need them. This notebook looks at three types of places that matter most to youth and families:

| Layer | Why it matters |
|-------|----------------|
| 🏫 **Schools** | Kids who age out of school bus coverage need transit to get there |
| 📚 **Libraries** | Free, public, heavily used by students and families, often after school |
| 🏛️ **Recreation centres** | Recreation, programs, and social infrastructure|

**The question:** How many of these schools sit  more than 500m  from the nearest Calgary Transit stop and 30 minutes from a social third place?

---

**Data sources (all free, open):**
- Calgary Transit stops: [Open Calgary - GTFS](https://data.calgary.ca/Transportation-Transit/Calgary-Transit-Scheduling-Data/npk7-z3bj) 
- Schools: [Open Calgary - Schools](https://data.calgary.ca/Services-and-Amenities/Schools/fd9t-tdn2)
- Libraries: [Open Calgary - Library Locations](https://data.calgary.ca/Recreation-and-Culture/Calgary-Public-Library-Locations-and-Hours/m9y7-ui7j)
- Recreation Centres: [Open Calgary - Recreation Centres](https://data.calgary.ca/Recreation-and-Culture/Recreation-Facilities-Map/vb9x-684u) 

## 1. Loading data

In [2]:
import pandas as pd
import numpy as np
from scipy.spatial import cKDTree
import folium

import heapq
from collections import defaultdict


#  Stops 
stops = pd.read_csv('stops.txt').dropna(subset=['stop_lat', 'stop_lon'])
stops = stops[['stop_id', 'stop_name', 'stop_lat', 'stop_lon']]
print(f"Stops: {len(stops)}\n")

#  Stop times 
stop_times = pd.read_csv('stop_times.txt', usecols=['trip_id', 'arrival_time', 'departure_time', 'stop_id', 'stop_sequence'])
print(f"Stop time: {len(stop_times)}\n")

#  Trips 
trips = pd.read_csv('trips.txt', usecols=['route_id', 'service_id', 'trip_id'])
print(f"Trips: {len(trips)}\n")

#  Calendar 
calendar = pd.read_csv('calendar.txt')
print(f"Calendar: {len(calendar)}\n")


Stops: 6173

Stop time: 3680719

Trips: 109748

Calendar: 31



In [3]:
# Schools 
schools = pd.read_csv('schools.csv')
coords = schools['POINT'].str.extract(r'POINT \(([\d.-]+) ([\d.-]+)\)')
schools['lon'] = coords[0].astype(float)
schools['lat'] = coords[1].astype(float)
schools['category'] = 'School'
schools.rename(columns={'NAME': 'name'}, inplace=True)
print(f"{len(schools)} Schools loaded")


# Libraries 
libraries = pd.read_csv('libraries.csv')
coords = libraries['Location'].str.extract(r'\(([\d.-]+),\s*([\d.-]+)\)')
libraries['lon'] = coords[0].astype(float)
libraries['lat'] = coords[1].astype(float)
libraries['category'] = 'Library'
libraries.rename(columns={'Library': 'name'}, inplace=True)
print(f"{len(libraries)} Libraries loaded")

#  Recreation Centres 
rec = pd.read_csv('rec.csv')
coords = rec['POINT'].str.extract(r'POINT \(([\d.-]+) ([\d.-]+)\)')
rec['lon'] = coords[0].astype(float)
rec['lat'] = coords[1].astype(float)
rec['category'] = 'Recreation Centre'
rec.rename(columns={'COMPLEX_NAME': 'name'}, inplace=True)
print(f"{len(rec)} Recreation loaded")


# Combine third spaces
third_spaces = pd.concat([libraries, rec], ignore_index=True)


print(f"\nSchools: {len(schools)}")
print(f"Third spaces (libraries + community centres): {len(third_spaces)}")

506 Schools loaded
22 Libraries loaded
112 Recreation loaded

Schools: 506
Third spaces (libraries + community centres): 134


## 2. Filter by After school times

Assuming most students would be hangning out after school, we want trips that run on **weekday evenings (3:30–10:00pm)**. GTFS stores times as `HH:MM:SS` strings  including values past midnight like `25:30:00` for overnight trips. We convert to total seconds for reliable comparison.

In [4]:
def time_to_seconds(t):
    """Convert GTFS HH:MM:SS (including >24h) to total seconds."""
    try:
        h, m, s = str(t).strip().split(':')
        return int(h) * 3600 + int(m) * 60 + int(s)
    except:
        return None
stop_times['dep_sec'] = stop_times['departure_time'].apply(time_to_seconds).astype(int) #convert to seconds

# Window: 3:30pm to 10:00pm
WINDOW_START = 15.5 * 3600 
WINDOW_END   = 22 * 3600   
st = stop_times[
    (stop_times['dep_sec'] >= WINDOW_START) &
    (stop_times['dep_sec'] <= WINDOW_END)
].copy()#filter stop times

print(f"Stop-time records of interest (330–10pm): {len(st)}")

# Weekday service IDs from calendar
weekday_services = calendar[
    (calendar['monday'] == 1) |
    (calendar['tuesday'] == 1) |
    (calendar['wednesday'] == 1) |
    (calendar['thursday'] == 1) |
    (calendar['friday'] == 1)
]['service_id'].unique()

weekday_trips = trips[trips['service_id'].isin(weekday_services)]['trip_id'].unique()

# Filter morning stop times to weekday trips only
st_sorted = st[st['trip_id'].isin(weekday_trips)]
print(f"After weekday filter: {len(st_sorted):,}")

Stop-time records of interest (330–10pm): 1288942
After weekday filter: 649,101


## 3. Build KD-Tree and radius lookup

We filter further with **all stops within 500m** of each location using `tree.query_ball_point()`. 
We do this for both schools and third spaces, giving us a full catchment on both ends of the journey

In [5]:
EARTH_RADIUS_M  = 6371000
WALK_THRESHOLD_M = 500

WALK_THRESHOLD_RAD = WALK_THRESHOLD_M / EARTH_RADIUS_M

# Build KD-Tree on all stops (in radians)
stop_coords_rad = np.radians(stops[['stop_lat', 'stop_lon']].values)
tree = cKDTree(stop_coords_rad)

def stops_within_radius(df):
    """
    For each row in df, return a list of stop_ids within WALK_THRESHOLD_M.
    Uses cKDTree.query_ball_point for efficient radius queries.
    Returns a list of lists (one per facility).
    """
    coords_rad = np.radians(df[['lat', 'lon']].values)
    # query_ball_point returns a list of index arrays  one per query point
    idx_lists = tree.query_ball_point(coords_rad, r=WALK_THRESHOLD_RAD)
    return [
        stops.iloc[idxs]['stop_id'].tolist() if len(idxs) > 0 else []
        for idxs in idx_lists
    ]

# Schools: all stops within 500m
schools['nearby_stop_ids'] = stops_within_radius(schools)
schools['has_stop_access'] = schools['nearby_stop_ids'].apply(lambda x: len(x) > 0)
schools['nearby_stop_count'] = schools['nearby_stop_ids'].apply(len)

# Third spaces: all stops within 500m
third_spaces = third_spaces.dropna(subset=['lat', 'lon']).reset_index(drop=True)
third_spaces['nearby_stop_ids'] = stops_within_radius(third_spaces)
third_spaces['has_stop_access'] = third_spaces['nearby_stop_ids'].apply(lambda x: len(x) > 0)

print(f"Schools with ≥1 stop within 500m: {schools['has_stop_access'].sum()} / {len(schools)}")
print(f"Third spaces with ≥1 stop within 500m: {third_spaces['has_stop_access'].sum()} / {len(third_spaces)}")
print(f"\nAvg nearby stops per school (where >0): {schools[schools['has_stop_access']]['nearby_stop_count'].mean():.1f}")

Schools with ≥1 stop within 500m: 490 / 506
Third spaces with ≥1 stop within 500m: 105 / 133

Avg nearby stops per school (where >0): 7.7


## 4. Build the reachability graph

For stops of interest , we record which other stops are reachable within 20 minutes on a single trip during the afternoon window.

`reachable_from[stop_id]` → `set` of stop_ids reachable within 20 min

Make the map

In [6]:
RIDE_LIMIT_SEC = 20 * 60 #limit by 20 min
#find shortest time for stop to stop; acts at cost in dji algo
edge_min = {}   # min ride (seconds) for a set of stops
 
for trip_id, group in st_sorted.groupby('trip_id'):
    seq = group.sort_values('stop_sequence')
    stops_in_trip = seq[['stop_id', 'dep_sec']].values
 
    for i in range(len(stops_in_trip) - 1):
        from_stop, t_board  = stops_in_trip[i]
        to_stop,   t_alight = stops_in_trip[i + 1]
        ride = int(t_alight) - int(t_board)
        if ride <= 0 or ride > 20*60:
            continue
        key = (from_stop, to_stop)
        if key not in edge_min or ride < edge_min[key]:
            edge_min[key] = ride
 
# Convert to adjacency list
adj = defaultdict(list)
for (strts, ends), w in edge_min.items():
    adj[strts].append((ends, w))
 
print(f"Graph nodes (stops with departures): {len(adj):,}")
print(f"Graph edges (stop→stop legs):        {len(edge_min):,}")
 


Graph nodes (stops with departures): 5,475
Graph edges (stop→stop legs):        6,347


Search the map for routes within our criteria


In [7]:
def dijkstra_reachable(origin_stop,end_stops, cutoff=RIDE_LIMIT_SEC):
    """
    Heap-based Dijkstra from origin_stop.
    Returns dict {stop_id: ride_seconds} for all end_stops stops within cutoff.
    The origin itself is excluded from the result.
    """
    dist = {origin_stop: 0}
    heap = [(0, origin_stop)]   # (cost, stop_id)
 
    while heap:
        cost, u = heapq.heappop(heap)
        if cost > dist.get(u, float('inf')):
            continue                         #bad data
        if cost > cutoff:
            break                             #  nothing cheaper remains; 20 min dead end
 
        for v, w in adj.get(u, []):
            new_cost = cost + w
            if new_cost <= cutoff and new_cost < dist.get(v, float('inf')):
                dist[v] = new_cost # min cost of stop
                if v not in end_stops:  # no need to explore beyond a rec centre
                    heapq.heappush(heap, (new_cost, v)) # keep exploring
 
    # Exclude the origin non end stops

    return {s: t for s, t in dist.items() if s != origin_stop and s in end_stops}
 
 
# find all stops from schools to third places
flat = []
for stop_list in schools['nearby_stop_ids']:
    for stop_id in stop_list:
        flat.append(stop_id)
all_school_stops = set(flat)
flat = []
for stop_list in third_spaces['nearby_stop_ids']:
    for stop_id in stop_list:
        flat.append(stop_id)
all_third_stops = set(flat)

#run dijkstra to find routes
reachable_from = {}
for i, stop_id in enumerate(all_school_stops):
    reachable_from[stop_id] = dijkstra_reachable(stop_id,all_third_stops)
 
print("Dijkstra complete.")
 

Dijkstra complete.


## 5. Check connectivity for each school and classify

For each school:
1. Take the **union** of all stops reachable within 20 min from any of its nearby stops
2. Check if that union intersects with the third-space entry stop set
3. If yes → **fully connected**


In [8]:

#get which third places are near which stop
stop_to_rec = defaultdict(list)
stop_to_lib = defaultdict(list)
for _, ts in third_spaces[third_spaces['has_stop_access']].iterrows():
    for stop_id in ts['nearby_stop_ids']:
        if ts['category'] == 'Recreation Centre':
            stop_to_rec[stop_id].append(ts['name'])
        else:
            stop_to_lib[stop_id].append(ts['name'])
def score_school(row):
    """
    Returns a dict with connectivity scores for one school row.
    """
    if not row['has_stop_access']:
        return {
            'has_stop': False, 'has_dest': False,
            'n_rec_reachable': 0, 'n_lib_reachable': 0,
            'rec_names': [], 'lib_names': [], 'reachable_spaces': [],'n_reachable_spaces': 0
        }
 
    # Union of all stops walkable to school
    all_walkable = set(row['nearby_stop_ids'])

    #get set of all third place stops reachable from stops
    flat_ts_reach =[]
    for os in all_walkable:
        flat_ts_reach.extend(reachable_from.get(os, {}))


    #count how many are unique third places
    rec_names = sorted({name for stop_id in flat_ts_reach for name in stop_to_rec[stop_id]})
    lib_names = sorted({name for stop_id in flat_ts_reach for name in stop_to_lib[stop_id]})
    
    return {
    'has_stop': True, 'has_dest': 1<(len(rec_names)+len(lib_names)),
    'n_rec_reachable': len(rec_names), 'n_lib_reachable': len(lib_names),
    'rec_names': rec_names, 'lib_names': lib_names,
    'reachable_spaces': (rec_names + lib_names),'n_reachable_spaces': (len(rec_names)+len(lib_names))
}
 
    #check origin of destination stop
    
        
 
scores = schools.apply(score_school, axis=1, result_type='expand')
schools = pd.concat([schools, scores], axis=1)
 
# Classification tier
def classify(row):
    if not row['has_stop_access']:
        return 'Transit desert'
    if not row['has_dest']:  
        return 'Stop access only'
    if  row['n_reachable_spaces']<3:
        return 'Connected'
    return 'Well Connected'
 
schools['connectivity'] = schools.apply(classify, axis=1)
 
print("\nConnectivity summary:")
print(schools['connectivity'].value_counts().to_string())
print(f"\nSchools with ≥3 rec centre reachable: {(schools['n_rec_reachable'] > 3).sum()/len(schools)}")
print(f"\nTop 10 schools by Third place centres reachable:")
print(
    schools[schools['n_reachable_spaces'] > 0]
    .sort_values('n_rec_reachable', ascending=False)
    [['name', 'n_reachable_spaces']]
    .head(10)
    .to_string(index=False)
)

 
 


Connectivity summary:
connectivity
Well Connected      266
Stop access only    129
Connected            95
Transit desert       16

Schools with ≥3 rec centre reachable: 0.42292490118577075

Top 10 schools by Third place centres reachable:
                               name  n_reachable_spaces
             Queen Elizabeth School                  19
        Queen Elizabeth High School                  19
          Chinook Learning Services                  19
                    Balmoral School                  19
               AADAC Youth Services                  18
                 Louise Dean School                  18
                   Hillhurst School                  17
                  University School                  17
          Westmount Mid/High School                  17
STEM Innovation Academy High School                  17


## 6. Classify schools

In [ ]:
import branca.colormap as cm

max_rec = schools['n_rec_reachable'].max()
colorscale = cm.linear.YlOrRd_09.scale(0, max(max_rec, 1))
 

 
m = folium.Map(location=[51.0447, -114.0719], zoom_start=11, tiles='CartoDB positron')
 
# Transit stops
stop_layer = folium.FeatureGroup(name='Transit stops', show=True)
for _, stop in stops.iterrows():
    folium.CircleMarker(
        location=[stop['stop_lat'], stop['stop_lon']],
        radius=2, color='#bbbbbb', fill=True, fill_opacity=0.3,
        tooltip=stop['stop_name']
    ).add_to(stop_layer)
stop_layer.add_to(m)
 
# Third spaces
ts_layer = folium.FeatureGroup(name='Libraries & Rec Centres', show=True)
for _, ts in third_spaces.iterrows():
    icon = 'book' if ts['category'] == 'Library' else 'futbol-o'
    folium.CircleMarker(
        location=[ts['lat'], ts['lon']],
        radius=2, color="#0000FF", fill=True, fill_opacity=0.3,
        tooltip=ts['name']
    ).add_to(ts_layer)
ts_layer.add_to(m)
 
# Schools
school_layer = folium.FeatureGroup(name='Schools (sized by rec centres reachable)', show=True)
 
schools_sorted = schools.sort_values('n_rec_reachable', ascending=False).reset_index(drop=True)
schools_sorted['rank'] = schools_sorted['n_rec_reachable'].rank(method='min', ascending=False).astype(int)
 
for _, school in schools_sorted.iterrows():
    rec_list  = ', '.join(school['rec_names'])  if school['rec_names']  else 'None'
    lib_list  = ', '.join(school['lib_names'])  if school['lib_names']  else 'None'
    rank_str  = f"#{int(school['rank'])}" if school['n_rec_reachable'] > 0 else "-"
 
    popup_html = f"""
        <b>{school['name']}</b><br>
        Status: <b>{school['connectivity']}</b><br>
        Rec-centre rank: <b>{rank_str}</b><br><br>
        Rec centres reachable (≤20 min): <b>{school['n_rec_reachable']}</b><br>
        <small>{rec_list}</small><br><br>
        Libraries reachable (≤20 min): <b>{school['n_lib_reachable']}</b><br>
        <small>{lib_list}</small><br><br>
        Walkable stops: <b>{school['nearby_stop_count']}</b> within 500 m
    """
 
    radius = 5 + min(school['n_rec_reachable'] * 1.5, 13)   # 5–18 px
    colour = colorscale(school['n_rec_reachable'])
 
    folium.CircleMarker(
        location=[school['lat'], school['lon']],
        radius=radius,
        color='#333333',
        weight=1,
        fill=True,
        fill_color=colour,
        fill_opacity=0.85,
        popup=folium.Popup(popup_html, max_width=300),
        tooltip=f"{school['name']} - {school['n_rec_reachable']} rec centres"
    ).add_to(school_layer)
 
school_layer.add_to(m)
folium.LayerControl(collapsed=False).add_to(m)

# Colour-scale legend (bottom-right)
colorscale.caption = 'Rec centres reachable within 20 min'
colorscale.add_to(m)
 
# Text legend (bottom-left)
legend_html = """
<div style="position:fixed; bottom:30px; left:30px; z-index:1000;
            background:white; padding:14px 18px; border-radius:8px;
            box-shadow:2px 2px 8px rgba(0,0,0,0.25);
            font-family:sans-serif; font-size:12px; max-width:240px;">
  <b style="font-size:13px;">How to read this map</b><br><br>
  <b>Circle size</b> = rec centres reachable in ≤20 min of riding<br>
  <b>Circle colour</b> = count (yellow → red scale)<br><br>
  
  <b>Methodology</b><br>
  School → bus stop ≤500 m walk<br>
  Dijkstra shortest path ≤20 min ride<br>
  (transfers allowed; ride-time only)<br>
  Rec/library stop ≤500 m walk<br>
  Time window: weekdays 3:30–10 pm
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))
 
m.save('calgary_kids_transit_ranked.html')
print("Map saved → calgary_kids_transit_ranked.html")


Map saved → calgary_kids_transit_ranked.html


## 7. Key findings


In [24]:
well     = schools[schools['connectivity'] == 'Well Connected']
conn     = schools[schools['connectivity'] == 'Connected']
stop_only = schools[schools['connectivity'] == 'Stop access only']
desert    = schools[schools['connectivity'] == 'Transit desert']
 
print("=" * 62)
print("KEY FINDINGS - Calgary School Transit Connectivity")
print("=" * 62)
print(f"Total schools analysed:      {len(schools)}")
print(f"Methodology: all stops within 500m on both ends, 20-min ride")
print()
print(f"Well connected:          {len(well):>4} ({len(well)/len(schools)*100:.1f}%)")
print(f"Connected:                {len(conn):>4} ({len(conn)/len(schools)*100:.1f}%)")
print(f"Stop access only:         {len(stop_only):>4} ({len(stop_only)/len(schools)*100:.1f}%)")
print(f"Transit desert:           {len(desert):>4} ({len(desert)/len(schools)*100:.1f}%)")
print()
print(f"Avg walkable stops per school (where >0): "
      f"{schools[schools['has_stop']]['nearby_stop_count'].mean():.1f}")
print(f"Median walkable stops per school (where >0): "
      f"{schools[schools['has_stop']]['nearby_stop_count'].median():.1f}")


KEY FINDINGS - Calgary School Transit Connectivity
Total schools analysed:      506
Methodology: all stops within 500m on both ends, 20-min ride

Well connected:           266 (52.6%)
Connected:                  95 (18.8%)
Stop access only:          129 (25.5%)
Transit desert:             16 (3.2%)

Avg walkable stops per school (where >0): 7.7
Median walkable stops per school (where >0): 7.0


# Does Calgary Have Transit Deserts?
### Measuring recreational access by transit for school-aged kids

---

## The Problem

As cities debate regulating social media for young people, there is little conversation about what social alternatives exist. Public transit is the primary way car-free youth get around independently, but stops aren't always where they need to be, and even when they are, the routes don't always go anywhere useful.

This project asks a specific, computable version of that question: **How many Calgary schools sit within walking distance of a transit stop, and from that stop, can a kid reach a recreation centre or library within 20 minutes on weekday afternoons?**

The answer is mapped, ranked, and broken into four tiers of connectivity. This should show how access to social third spaces spans across the city, and room for improvement.

---

## Data

All data is free, open and provided by the city of Calgary.

| Dataset | Source | Notes |
|---------|--------|-------|
| Calgary Transit stops & schedules | Open Calgary (GTFS) | stops.txt, stop_times.txt, trips.txt, calendar.txt |
| Schools | Open Calgary | Coordinates stored as WKT `POINT(lon lat)` strings |
| Libraries | Open Calgary | Coordinates stored as `(lat, lon)` strings |
| Recreation centres | Open Calgary | Coordinates stored as WKT `POINT(lon lat)` strings |

The coordinate formats were inconsistent across datasets and required separate regex parsers for each. GTFS departure times are stored as `HH:MM:SS` strings including values past midnight so all times were converted to total seconds to make comparisons reliable. Additionally, some data was missing and malformed so the data was cleaned. 

---

## Methodology

### Step 1 - Filter to relevant trips

Due to the complexity of the transit system and the large amount of data, I quickly narrowed down the focus of the analysis to target weekday afternoons from 3:30pm to 10:00pm, when kids are most likely traveling independently after school. Trips were filtered to this window and to weekday service IDs from the GTFS calendar. 

### Step 2 - Spatial radius lookup with a KD-Tree

Rather than computing the distance between every school and every stop (an O(n²) operation), a KD-Tree was built on all stop coordinates converted to radians. `query_ball_point()` then returns all stops within 500m of each school and each third space in a single vectorised call. This runs in roughly O(n log n) and handles hundreds of schools and thousands of stops without looping.

The 500m threshold represents a reasonable walking distance. Both ends of the journey were indexed; stops near schools (departure) and stops near libraries and rec centres (arrival). This further cut down the number of AI computation need to be calculated.

### Step 3 - Build a weighted transit graph

The GTFS schedule was converted into a directed weighted graph where each node is a stop and each edge is a direct bus leg between consecutive stops, weighted by ride time in seconds.

Because the same stop pair can appear on multiple trips at different times, only the minimum ride time across all trips was kept per stop pair. This answers "what is the fastest this leg could possibly be?" rather than "when does the next bus leave?".The graph was stored as an adjacency list (`defaultdict(list)`) so that neighbour lookups during pathfinding are O(1) rather than requiring a full table scan.

Dijkstra's algorithm was run from every stop near a school, finding all third-space entry stops reachable within 20 minutes of total ride time. BFS was considered but rejected because BFS only finds shortest paths when all edges are equal weight and requires the whole graph to be scanned. Here legs have different durations, so BFS would return incorrect results.

Transfers between routes are handled automatically. Dijkstra doesn't track which route an edge belongs to, only the cumulative cost. A path that takes route 3 for 8 minutes and route 17 for 9 minutes has a total cost of 17 minutes and is included. Transfer wait time is not modelled, this is a limitation discussed below.

Two optimizations were applied to keep the search efficient:

- The heap is broken out of early once `cost > cutoff`, since the heap is sorted and nothing cheaper can remain
- Once Dijkstra reaches a third-space entry stop it records it but does not push it onto the heap, since there is no need to explore onward from a destination

```python
if v not in end_stops:
    heapq.heappush(heap, (new_cost, v))  # only keep exploring from non-destinations
```

Results were stored as `reachable_from[stop_id]`, a dict of `{third_space_stop: seconds}`, for every school-adjacent stop.

### Step 5 - Score each school

For each school, the reachable third-space stops from all of its nearby stops were unioned together. A reverse lookup dict (`stop_to_rec`, `stop_to_lib`) was pre-built outside the scoring function so that translating a stop ID to a rec centre or library name is an O(1) dict lookup rather than a DataFrame filter per school.


### Step 6 - Four-tier classification

Schools were classified into four tiers:

| Tier | Criteria |
|------|----------|
|  Transit desert | No stop within 500m |
|  Stop access only | Stop nearby, but no third space reachable in 20 min |
|  Connected | 1–2 third spaces reachable |
|  Well connected | 3 or more third spaces reachable |

---

## Modelling Decisions  

An earlier approach stored every departure time for every stop pair and attempted to simulate exact boarding sequences. This added significant complexity, tracking when a kid arrives at a stop, whether they missed a departure, and when the next one comes, without meaningfully improving the question being answered. Since the goal is "can a kid get there within 20 minutes?" rather than "when exactly does the next bus leave?", departure times are out of scope. The schedule was collapsed to a single minimum-cost edge per stop pair across all trips:

```
pythonkey = (from_stop, to_stop)
if key not in edge_min or ride < edge_min[key]:
    edge_min[key] = ride
```
This answers the capability question directly, if the fastest version of a leg fits within 20 minutes, the destination is reachable. It also reduced the graph to its smallest useful form, cutting Dijkstra's search space significantly. The tradeoff is that it assumes best-case headways, which is noted in the limitations.
--- 
## Limitations

**Transfer wait time is not modelled.** The 20-minute cap applies to pure ride time. In practice, a transfer adds waiting time that could push a journey well past 20 minutes. A more realistic model would add a penalty per transfer. Using an algorithm like RAPTOR would provide more realistic travel time estimates, but requires significantly more computation.

**Walking speed is assumed.** The 500m radius assumes a comfortable walking pace. For younger children, or in winter conditions, a tighter radius (300–400m) may be more realistic.

**Best-case headways.** Using the minimum ride time across all trips assumes the fastest possible leg is always available. In reality a kid might wait 15 minutes for a connection. This analysis measures transit capability, not typical experience.

**Scope of third spaces.** This model only considers city-owned recreation centres and libraries. It does not account for private facilities, extracurricular activities, or pre-scheduled commitments that might change where a kid is actually trying to go.

**Boarding stop assumed.** The model assumes a kid boards at whichever nearby stop gives the best connection. It does not account for stop familiarity or whether a particular stop feels safe or accessible.

---

## Results

Schools in and around downtown Calgary are the most connected, reflecting the density of the transit network there. Schools on the suburban areas, particularly in newer neighbourhoods in the south and northeast, show the weakest connectivity. Regardless, the average school has access to three or more recreational facilities, meaning kids should have their pick of third spaces. Not it is up to libraries, YMCAs and parks to provide programming to keep them engaged. 

---

## Skills Demonstrated

- Spatial indexing with `scipy.spatial.cKDTree`
- Graph construction and shortest-path search (Dijkstra) on real GTFS transit data
- GTFS data parsing including edge cases (>24h time strings, inconsistent coordinate formats)
- Pandas data pipeline with `apply`, `defaultdict` lookups, and set operations
- Interactive map with ranked, layered visualisation using Folium and Branca